# Week 5 · Day 2 — pandas I: `loans_status.csv`

Bekzod's request: *the same portfolio numbers as week 4, in a notebook I can re-run every month.*
The first cells are the walk-through from the page (run them, read the output); the TODO cells are yours.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
DATA = Path("..")              # this notebook lives in data/python/, so ".." is the data/ folder
AS_OF = pd.Timestamp("2026-10-01")

In [2]:
loans = pd.read_csv(DATA / "excel" / "loans_status.csv", parse_dates=["disbursed_at"])
loans.shape

(7828, 13)

In [3]:
loans.head()

,loan_id,customer,region,product,branch,grade,disbursed_at,principal,term,status,outstanding,dpd,bucket
0,1,Gulbahor Sobirova,Termez,Consumer loan,Termez Central,C,2025-03-28,31150000,24,active,9593064,0,0
1,2,Nilufar Mirzaeva,Nukus,Consumer loan,Nukus Central,A,2025-12-05,40060000,18,active,22246389,0,0
2,3,Sardor Abdullaev,Bukhara,Micro loan,Bukhara Central,A,2026-03-28,1590000,3,closed,0,0,0
3,4,Kamol Karimov,Tashkent,Micro loan,Tashkent Central,B,2026-08-23,2520000,3,active,1708700,0,0
4,5,Shahnoza Saidova,Namangan,Small business loan,Namangan Central,A,2024-05-22,25590000,36,active,7498297,0,0


In [4]:
loans.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7828 entries, 0 to 7827
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   loan_id       7828 non-null   int64         
 1   customer      7828 non-null   object        
 2   region        7828 non-null   object        
 3   product       7828 non-null   object        
 4   branch        7828 non-null   object        
 5   grade         7828 non-null   object        
 6   disbursed_at  7828 non-null   datetime64[ns]
 7   principal     7828 non-null   int64         
 8   term          7828 non-null   int64         
 9   status        7828 non-null   object        
 10  outstanding   7828 non-null   int64         
 11  dpd           7828 non-null   int64         
 12  bucket        7828 non-null   object        
dtypes: datetime64[ns](1), int64(5), object(7)
memory usage: 795.2+ KB


In [5]:
loans.describe()

,loan_id,disbursed_at,principal,term,outstanding,dpd
count,7828.000000,7828,7.828000e+03,7828.000000,7.828000e+03,7828.000000
mean,3956.596449,2025-08-27 22:34:27.654573312,5.170967e+07,11.777338,2.666371e+07,12.560296
min,1.000000,2024-01-02 00:00:00,3.000000e+05,3.000000,0.000000e+00,0.000000
25%,1976.750000,2025-02-03 18:00:00,2.880000e+06,6.000000,0.000000e+00,0.000000
50%,3950.500000,2025-10-03 00:00:00,5.790000e+06,6.000000,0.000000e+00,0.000000
75%,5936.250000,2026-04-22 00:00:00,3.471250e+07,18.000000,7.552471e+06,0.000000
max,7917.000000,2026-10-01 00:00:00,5.990600e+08,48.000000,5.860687e+08,919.000000
std,2286.054065,NaN,1.125922e+08,10.180578,7.739826e+07,78.730804


## Selecting and filtering — every `WHERE` you know, as a boolean mask

In [6]:
active = loans[loans["status"] == "active"]
late = loans[(loans["dpd"] > 0) & (loans["status"] == "active")]
big = loans[loans["product"].isin(["Small business loan", "Auto loan"])]
tashkent = loans[loans["branch"].str.contains("Tashkent")]
mid = loans[loans["principal"].between(10_000_000, 50_000_000)]
print(len(active), len(late), len(big), len(tashkent), len(mid))

3352 151 1172 3020 2087


In [7]:
loans.loc[loans["dpd"] > 90, ["loan_id", "product", "dpd", "outstanding"]].sort_values("dpd", ascending=False).head()

,loan_id,product,dpd,outstanding
2145,2167,Micro loan,919,0
7653,7741,Consumer loan,916,0
363,368,Buy now pay later,904,0
7255,7339,Micro loan,902,0
5506,5571,Micro loan,888,0


## Dates, new columns, value_counts

In [8]:
loans["year"] = loans["disbursed_at"].dt.year
loans["month"] = loans["disbursed_at"].dt.to_period("M")
y2026 = loans[(loans["disbursed_at"] >= "2026-01-01") & (loans["disbursed_at"] < "2026-10-01")]
print(len(y2026), "loans in 2026 (to 30 Sep),", y2026["principal"].sum(), "UZS")
loans["status"].value_counts()

3030 loans in 2026 (to 30 Sep), 161480300000 UZS


status
closed         4242
active         3352
written_off     190
default          44
Name: count, dtype: int64

## `pd.cut` — the buckets without a CASE

In [9]:
edges = [-1, 0, 30, 60, 90, np.inf]
labels = ["0", "1-30", "31-60", "61-90", "90+"]
loans["bucket_pd"] = pd.cut(loans["dpd"], bins=edges, labels=labels)
loans["bucket_pd"].value_counts().sort_index()

bucket_pd
0        7429
1-30      124
31-60      25
61-90      16
90+       234
Name: count, dtype: int64

In [10]:
assert (loans["bucket_pd"].astype(str) == loans["bucket"]).all(), "pd.cut disagrees with the file's bucket column"
print("buckets match the SQL column")

buckets match the SQL column


## TODO — `portfolio_summary(df)`: the six tiles
Return a dict with keys `loans`, `active`, `outstanding`, `disbursed_2026`, `par30_pct`, `npl90_pct` (percentages rounded to 2 decimals;
`disbursed_2026` counts loans disbursed from 2026-01-01 up to but not including 2026-10-01). The assert cell knows week 3's numbers.

In [11]:
def portfolio_summary(df):
    """Six portfolio tiles from a loans_status frame, as a dict."""
    raise NotImplementedError("TODO: write me, then run the assert cell below")

In [12]:
t = portfolio_summary(loans)
assert t["loans"] == 7828 and t["active"] == 3352
assert t["outstanding"] == 208_723_504_430
assert t["disbursed_2026"] == 3030
assert t["par30_pct"] == 3.12 and t["npl90_pct"] == 2.55
t

NotImplementedError: TODO: write me, then run the assert cell below

## TODO — write the month's file
Save a frame with one row per bucket (`bucket`, `loans`, `outstanding`, `share_pct`) to `portfolio_buckets.csv` next to this notebook, no index column.
Open it in Excel: that is Bekzod's refreshable table.

In [13]:
# your code here